# RoboRAG — Robotics Course Document Assistant

## RAG Pipeline Development and Evaluation

RoboRAG is a Retrieval-Augmented Generation (RAG) system designed to answer questions about robotics course materials.

The knowledge base consists of robotics lecture PDFs covering topics such as:

- Introduction to Robotics
- Rigid Motion
- 3D Rotation
- Forward Kinematics
- Velocity Kinematics
- Jacobians
- Singularities
- Mobile Robots

The system will:

1. Extract text from the source PDFs
2. Clean and split the text into chunks
3. Generate vector embeddings
4. Store the embeddings in ChromaDB
5. Retrieve relevant chunks for a user question
6. Provide the retrieved context to a local Ollama LLM
7. Generate grounded answers with source citations
8. Evaluate the pipeline using robotics questions


In [1]:
import sys

print(sys.version)
print(sys.executable)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
c:\Users\yasser\Projects\rag-study-assistant\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import sys

expected = Path(
    r"C:\Users\yasser\Projects\rag-study-assistant\.venv\Scripts\python.exe"
)

actual = Path(sys.executable)

print("Correct environment:", actual == expected)

Correct environment: True


## Phase 2.1 — Load & Inspect ##

Cell 2 — Imports and configuration

In [3]:
from pathlib import Path
from pypdf import PdfReader
import pandas as pd
import numpy as np

print("Imports successful")

Imports successful


Cell 3 — Find our project folders

In [4]:
# Detect project root safely whether the notebook starts
# from the project root or the notebooks folder.

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Data directory exists:", DATA_DIR.exists())

Project root: c:\Users\yasser\Projects\rag-study-assistant
Data directory: c:\Users\yasser\Projects\rag-study-assistant\data\raw
Data directory exists: True


Cell 4 — Find the PDFs

In [5]:
pdf_files = sorted(DATA_DIR.glob("*.pdf"))

print(f"Number of PDFs found: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

Number of PDFs found: 11
01_intro_to_robotics.pdf
02_rigid_motion_1.pdf
03_rigid_motion_2.pdf
04_3d_rotation.pdf
05_forward_kinematics_1.pdf
06_forward_kinematics_2.pdf
07_velocity_kinematics.pdf
08_jacobian.pdf
09_singularities.pdf
10_mobile_robots_1.pdf
11_mobile_robots_2.pdf


Cell 5 — Load every page + metadata

In [6]:
documents = []
file_statistics = []
failed_files = []

for pdf_path in pdf_files:
    try:
        reader = PdfReader(pdf_path)

        text_pages = 0
        empty_pages = 0
        total_characters = 0

        for page_number, page in enumerate(reader.pages, start=1):

            text = page.extract_text() or ""
            text = text.strip()

            if text:
                text_pages += 1
                total_characters += len(text)
            else:
                empty_pages += 1

            documents.append(
                {
                    "text": text,
                    "source": pdf_path.name,
                    "page": page_number
                }
            )

        file_statistics.append(
            {
                "file": pdf_path.name,
                "pages": len(reader.pages),
                "text_pages": text_pages,
                "empty_pages": empty_pages,
                "characters": total_characters
            }
        )

    except Exception as e:
        failed_files.append(
            {
                "file": pdf_path.name,
                "error": str(e)
            }
        )

print("Loading finished.")

Loading finished.


Cell 6 — Dataset statistics

In [7]:
stats_df = pd.DataFrame(file_statistics)

stats_df

,file,pages,text_pages,empty_pages,characters
0,01_intro_to_robotics.pdf,41,41,0,11931
1,02_rigid_motion_1.pdf,31,31,0,8743
2,03_rigid_motion_2.pdf,26,26,0,9465
3,04_3d_rotation.pdf,49,49,0,16749
4,05_forward_kinematics_1.pdf,36,36,0,13166
5,06_forward_kinematics_2.pdf,17,17,0,1780
6,07_velocity_kinematics.pdf,34,34,0,11093
7,08_jacobian.pdf,34,34,0,11093
8,09_singularities.pdf,24,24,0,8065
9,10_mobile_robots_1.pdf,53,53,0,18813


In [8]:
total_documents = len(pdf_files)
total_pages = sum(item["pages"] for item in file_statistics)
total_text_pages = sum(item["text_pages"] for item in file_statistics)
total_empty_pages = sum(item["empty_pages"] for item in file_statistics)

print("Dataset Summary")
print("----------------")
print(f"Documents: {total_documents}")
print(f"Total pages: {total_pages}")
print(f"Pages with extractable text: {total_text_pages}")
print(f"Pages without extractable text: {total_empty_pages}")
print(f"Failed files: {len(failed_files)}")

Dataset Summary
----------------
Documents: 11
Total pages: 418
Pages with extractable text: 418
Pages without extractable text: 0
Failed files: 0


Cell 7 — Check failed files

In [9]:
if failed_files:
    print("Files that failed to parse:")
    
    for item in failed_files:
        print(item)
else:
    print("All PDF files parsed successfully.")

All PDF files parsed successfully.


Cell 8 — Actually inspect extracted text

In [10]:
for doc in documents[:5]:
    print("=" * 80)
    print("SOURCE:", doc["source"])
    print("PAGE:", doc["page"])
    print()
    print(doc["text"][:500])
    print()

SOURCE: 01_intro_to_robotics.pdf
PAGE: 1

CSE 432 Robotics
Course Introduction
Ahmed Asker, Ph.D.
Associate Professor in Mechatronic and Robotics
Ahmed.asker@ejust.edu.eg
Egypt-Japan University of Science and Technology 
1

SOURCE: 01_intro_to_robotics.pdf
PAGE: 2

Dr. Ahmed Asker
Course Contents
• Introduction to robotics
• Rigid motions and homogeneous transformations
• Forward kinematics
• Inverse kinematics
• Differential kinematics
• Path and trajectory planning
2CSE 432 Robotics

SOURCE: 01_intro_to_robotics.pdf
PAGE: 3

Dr. Ahmed Asker
Reference books
• Robot Modeling and Control
3
 Robotics, Vision and Control
CSE 432 Robotics

SOURCE: 01_intro_to_robotics.pdf
PAGE: 4

Dr. Ahmed Asker
Introduction
• Robotics is a relatively young field of modern technology that 
crosses traditional engineering boundaries. 
• Understanding the complexity of robots and their applications 
requires knowledge of:
• Electrical engineering.
• Mechanical engineering.
• Systems and industrial engineer

## 2.1 Load and Inspect

The RoboRAG knowledge base consists of **11 PDF lecture documents** covering major robotics topics including rigid motion, 3D rotations, forward kinematics, velocity kinematics, Jacobians, singularities, and mobile robots.

The documents were parsed using `PyPDF`.

### Dataset Inspection

- Number of source documents: **11**
- File format: **PDF**
- Total number of pages: **418**
- Pages containing extractable text: **418**
- Pages without extractable text: **0**
- Files that failed to parse: **0**
- OCR required: **No**

All source documents were successfully parsed, and every page contained extractable text. Therefore, no OCR preprocessing was required.

The lecture PDFs provide sufficient textual content for building the RAG pipeline, while also covering a broad range of robotics topics suitable for retrieval and question answering.

## Phase 2.2 — Cleaning + Chunking Strategy

## 2.2 Text Cleaning and Chunking Strategy

Before generating embeddings, the extracted text is lightly cleaned to remove unnecessary whitespace while preserving the original technical content.

The cleaning process:
- removes repeated spaces and tabs,
- removes excessive blank lines,
- trims leading and trailing whitespace,
- preserves technical terminology, equations, symbols, and punctuation.

Aggressive cleaning is intentionally avoided because mathematical notation and robotics terminology may contain meaningful symbols.

In [11]:
import re

def clean_text(text):
    """
    Light text cleaning while preserving technical content.
    """
    if not text:
        return ""

    # Replace tabs with spaces
    text = text.replace("\t", " ")

    # Remove repeated spaces
    text = re.sub(r" +", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n\s*\n+", "\n", text)

    # Remove unnecessary spaces around newlines
    text = "\n".join(line.strip() for line in text.splitlines())

    return text.strip()

In [12]:
#Step 2.2.2 — Apply cleaning
cleaned_documents = []

for doc in documents:
    cleaned = clean_text(doc["text"])

    if cleaned:
        cleaned_documents.append(
            {
                "text": cleaned,
                "source": doc["source"],
                "page": doc["page"]
            }
        )

print("Original page records:", len(documents))
print("Cleaned page records:", len(cleaned_documents))

Original page records: 418
Cleaned page records: 418


In [13]:
#Step 2.2.3 — Compare before and after
print("BEFORE CLEANING:")
print("-" * 80)
print(documents[0]["text"][:1000])

print("\nAFTER CLEANING:")
print("-" * 80)
print(cleaned_documents[0]["text"][:1000])

BEFORE CLEANING:
--------------------------------------------------------------------------------
CSE 432 Robotics
Course Introduction
Ahmed Asker, Ph.D.
Associate Professor in Mechatronic and Robotics
Ahmed.asker@ejust.edu.eg
Egypt-Japan University of Science and Technology 
1

AFTER CLEANING:
--------------------------------------------------------------------------------
CSE 432 Robotics
Course Introduction
Ahmed Asker, Ph.D.
Associate Professor in Mechatronic and Robotics
Ahmed.asker@ejust.edu.eg
Egypt-Japan University of Science and Technology
1


In [14]:
#Step 2.2.4 — Define chunk settings
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)

Chunk size: 800
Chunk overlap: 150


In [15]:
#Step 2.2.6 — Write the chunking function
def chunk_text(text, chunk_size=800, overlap=150):
    """
    Split text into overlapping character-based chunks.
    """

    if not text:
        return []

    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")

    # Short text remains as one chunk
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        # Stop if we reached the end
        if end >= len(text):
            break

        start += chunk_size - overlap

    return chunks

In [16]:
#Step 2.2.7 — Create all chunks
chunks = []

for doc in cleaned_documents:

    page_chunks = chunk_text(
        doc["text"],
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )

    for chunk_index, chunk in enumerate(page_chunks):

        chunks.append(
            {
                "chunk_id": f"{doc['source']}_p{doc['page']}_c{chunk_index}",
                "text": chunk,
                "source": doc["source"],
                "page": doc["page"],
                "chunk_index": chunk_index
            }
        )

print("Total chunks created:", len(chunks))

Total chunks created: 422


In [17]:
long_pages = [
    doc for doc in cleaned_documents
    if len(doc["text"]) > CHUNK_SIZE
]

print("Total pages:", len(cleaned_documents))
print("Pages longer than 800 characters:", len(long_pages))

for doc in long_pages[:10]:
    print(
        doc["source"],
        "Page", doc["page"],
        "->", len(doc["text"]), "characters"
    )

Total pages: 418
Pages longer than 800 characters: 4
05_forward_kinematics_1.pdf Page 4 -> 832 characters
05_forward_kinematics_1.pdf Page 6 -> 801 characters
05_forward_kinematics_1.pdf Page 23 -> 828 characters
05_forward_kinematics_1.pdf Page 26 -> 854 characters


In [18]:
chunk_lengths = [len(chunk["text"]) for chunk in chunks]

print("Chunk Statistics")
print("----------------")
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {np.mean(chunk_lengths):.2f} characters")
print(f"Minimum chunk length: {np.min(chunk_lengths)} characters")
print(f"Maximum chunk length: {np.max(chunk_lengths)} characters")

Chunk Statistics
----------------
Total chunks: 422
Average chunk length: 337.99 characters
Minimum chunk length: 13 characters
Maximum chunk length: 800 characters


In [19]:
short_chunks = [
    chunk for chunk in chunks
    if len(chunk["text"]) < 100
]

print("Chunks shorter than 100 characters:", len(short_chunks))

for chunk in short_chunks:
    print("=" * 80)
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Length:", len(chunk["text"]))
    print("Text:", repr(chunk["text"]))

Chunks shorter than 100 characters: 59
Source: 01_intro_to_robotics.pdf
Page: 12
Length: 52
Text: 'Dr. Ahmed Asker\nIndustrial Robots\n13CSE 432 Robotics'
Source: 01_intro_to_robotics.pdf
Page: 13
Length: 48
Text: 'Dr. Ahmed Asker\nField Robots\n14\nCSE 432 Robotics'
Source: 01_intro_to_robotics.pdf
Page: 16
Length: 75
Text: 'Dr. Ahmed Asker\nSurgical Robot\n17\nCSE 432 Robotics\nda Vinci Surgical System'
Source: 01_intro_to_robotics.pdf
Page: 18
Length: 50
Text: 'Dr. Ahmed Asker\nType of Joints\n19\nCSE 432 Robotics'
Source: 01_intro_to_robotics.pdf
Page: 22
Length: 65
Text: 'Dr. Ahmed Asker\nArticulated manipulator (RRR)\n23\nCSE 432 Robotics'
Source: 01_intro_to_robotics.pdf
Page: 24
Length: 63
Text: 'Dr. Ahmed Asker\nSpherical Manipulator (RRP)\n25\nCSE 432 Robotics'
Source: 01_intro_to_robotics.pdf
Page: 25
Length: 59
Text: 'Dr. Ahmed Asker\nSCARA Manipulator (RRP)\n26\nCSE 432 Robotics'
Source: 01_intro_to_robotics.pdf
Page: 26
Length: 78
Text: 'Dr. Ahmed Asker\nCylindrical Manip

In [20]:
#Step 2.2.8 — Inspect some chunks
for chunk in chunks[:5]:
    print("=" * 80)
    print("Chunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Chunk index:", chunk["chunk_index"])
    print("Characters:", len(chunk["text"]))
    print()
    print(chunk["text"][:800])
    print()

Chunk ID: 01_intro_to_robotics.pdf_p1_c0
Source: 01_intro_to_robotics.pdf
Page: 1
Chunk index: 0
Characters: 179

CSE 432 Robotics
Course Introduction
Ahmed Asker, Ph.D.
Associate Professor in Mechatronic and Robotics
Ahmed.asker@ejust.edu.eg
Egypt-Japan University of Science and Technology
1

Chunk ID: 01_intro_to_robotics.pdf_p2_c0
Source: 01_intro_to_robotics.pdf
Page: 2
Chunk index: 0
Characters: 223

Dr. Ahmed Asker
Course Contents
• Introduction to robotics
• Rigid motions and homogeneous transformations
• Forward kinematics
• Inverse kinematics
• Differential kinematics
• Path and trajectory planning
2CSE 432 Robotics

Chunk ID: 01_intro_to_robotics.pdf_p3_c0
Source: 01_intro_to_robotics.pdf
Page: 3
Chunk index: 0
Characters: 110

Dr. Ahmed Asker
Reference books
• Robot Modeling and Control
3
 Robotics, Vision and Control
CSE 432 Robotics

Chunk ID: 01_intro_to_robotics.pdf_p4_c0
Source: 01_intro_to_robotics.pdf
Page: 4
Chunk index: 0
Characters: 382

Dr. Ahmed Asker
Introducti

In [21]:
#Step 2.2.9 — Calculate chunk statistics
chunk_lengths = [len(chunk["text"]) for chunk in chunks]

print("Chunk Statistics")
print("----------------")
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {np.mean(chunk_lengths):.2f} characters")
print(f"Minimum chunk length: {np.min(chunk_lengths)} characters")
print(f"Maximum chunk length: {np.max(chunk_lengths)} characters")

Chunk Statistics
----------------
Total chunks: 422
Average chunk length: 337.99 characters
Minimum chunk length: 13 characters
Maximum chunk length: 800 characters


In [22]:
#Step 2.2.10 — See chunk counts by PDF
chunks_df = pd.DataFrame(chunks)

chunk_counts = (
    chunks_df.groupby("source")
    .size()
    .reset_index(name="number_of_chunks")
)

chunk_counts

,source,number_of_chunks
0,01_intro_to_robotics.pdf,41
1,02_rigid_motion_1.pdf,31
2,03_rigid_motion_2.pdf,26
3,04_3d_rotation.pdf,49
4,05_forward_kinematics_1.pdf,40
5,06_forward_kinematics_2.pdf,17
6,07_velocity_kinematics.pdf,34
7,08_jacobian.pdf,34
8,09_singularities.pdf,24
9,10_mobile_robots_1.pdf,53


### Chunking Strategy

A **page-aware fixed-size chunking strategy** was used for the RoboRAG knowledge base.

Each PDF page is treated as an independent unit first so that the original source filename and page number can be preserved. This is important for later retrieval and citation generation.

Pages shorter than **800 characters** are kept as a single chunk. Pages longer than 800 characters are divided into overlapping chunks of approximately **800 characters** with an overlap of **150 characters**.

This strategy was selected because the source documents are mainly robotics lecture slides. Most slides contain a relatively small amount of text and usually focus on one concept, so keeping short pages intact helps preserve their meaning. Only a few pages contain enough text to require further splitting.

The overlap of 150 characters helps reduce information loss when an explanation crosses a chunk boundary.

### Chunking Results

- Total source pages: **418**
- Total chunks created: **422**
- Average chunk length: **337.99 characters**
- Minimum chunk length: **13 characters**
- Maximum chunk length: **800 characters**
- Chunk size: **800 characters**
- Chunk overlap: **150 characters**

Most pages resulted in one chunk. The main exception was `05_forward_kinematics_1.pdf`, which produced **40 chunks from 36 pages** because several pages exceeded the selected chunk-size limit and therefore required splitting.

Metadata preserved for every chunk:

- source document name
- page number
- chunk index
- unique chunk ID

This metadata will later be used to provide grounded answers with document and page citations.

## Phase 2.3 — Embeddings & Vector Store,

## 2.3 Embeddings and Vector Representation

After preprocessing and chunking, each text chunk is converted into a numerical vector representation called an **embedding**.

Embeddings capture the semantic meaning of text, allowing chunks with similar meanings to be located using vector similarity rather than exact keyword matching.

For RoboRAG, the `all-MiniLM-L6-v2` model from the Sentence Transformers library is used.

This model was selected because it is lightweight, efficient for local execution, and well suited to semantic search and retrieval tasks.

Each chunk is embedded independently while its original metadata, including source document, page number, and chunk ID, is preserved for later retrieval and citation.

In [23]:
#Step 2.3.2 — Import SentenceTransformer
from sentence_transformers import SentenceTransformer
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.0+cpu
CUDA available: False


In [24]:
#Step 2.3.3 — Load the embedding model
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print("Embedding model loaded:")
print(EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded:
sentence-transformers/all-MiniLM-L6-v2


In [25]:
#Step 2.3.4 — Prepare the chunk texts
{
    "chunk_id": "...",
    "text": "...",
    "source": "...",
    "page": 12
}

{'chunk_id': '...', 'text': '...', 'source': '...', 'page': 12}

In [26]:
chunk_texts = [chunk["text"] for chunk in chunks]

print("Number of texts to embed:", len(chunk_texts))
print()
print("Example chunk:")
print(chunk_texts[0][:500])

Number of texts to embed: 422

Example chunk:
CSE 432 Robotics
Course Introduction
Ahmed Asker, Ph.D.
Associate Professor in Mechatronic and Robotics
Ahmed.asker@ejust.edu.eg
Egypt-Japan University of Science and Technology
1


In [27]:
#Step 2.3.5 — Generate the embeddings
embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

In [28]:
#Step 2.3.6 — Inspect the embedding result
print("Embedding array shape:", 384)
print("Data type:", embeddings.dtype)

Embedding array shape: 384
Data type: float32


In [29]:
#Step 2.3.7 — Look at one embedding
print("First embedding:")
print(embeddings[0])

print()
print("Embedding dimensions:", len(embeddings[0]))

First embedding:
[-3.61223109e-02  3.00778868e-03  1.59753170e-02 -5.87787963e-02
 -2.87641380e-02 -7.35006779e-02  3.16457450e-02  6.28713593e-02
 -4.90393341e-02  7.78454840e-02  1.24273608e-02 -1.33049283e-02
  8.81522745e-02 -2.62260586e-02  1.44657297e-02  2.42336262e-02
 -2.07797103e-02 -7.62785077e-02  6.97964279e-04 -9.36547518e-02
  1.01994075e-01  9.47932154e-03  6.11285679e-02 -2.13238876e-02
 -1.21015668e-01  5.61690517e-02  1.37978466e-02  1.42751615e-02
 -9.00072139e-03 -8.58685300e-02 -1.33948808e-03  3.45400870e-02
 -4.11457717e-02 -2.05638688e-02 -2.41922494e-02  2.25740150e-02
 -2.20599901e-02 -9.77554694e-02 -3.36131789e-02  8.65898747e-03
  2.46172920e-02  1.02648539e-02  7.17863813e-02 -5.24295717e-02
  4.61079590e-02  5.80786876e-02  4.63984795e-02 -1.00027241e-01
  3.01927142e-02  6.94307219e-03 -1.02648310e-01 -4.98957857e-02
 -3.44017171e-04 -4.81335036e-02 -2.96945553e-02  2.69410275e-02
  8.75297710e-02  4.60522734e-02  7.55273178e-02 -1.14977390e-01
  9.3839

In [30]:
#Step 2.3.8 — Verify one embedding per chunk
assert len(embeddings) == len(chunks)

print("Chunks:", len(chunks))
print("Embeddings:", len(embeddings))
print("One embedding per chunk: VERIFIED")

Chunks: 422
Embeddings: 422
One embedding per chunk: VERIFIED


In [31]:
#Step 2.3.9 — Test whether semantic similarity actually works
from sentence_transformers.util import cos_sim

test_sentences = [
    "The Jacobian relates joint velocity to end-effector velocity.",
    "A Jacobian matrix connects joint rates with robot motion.",
    "Mobile robots can use differential drive wheels."
]

test_embeddings = embedding_model.encode(
    test_sentences,
    normalize_embeddings=True
)

similarities = cos_sim(test_embeddings, test_embeddings)

print(similarities)

tensor([[1.0000, 0.6535, 0.1446],
        [0.6535, 1.0000, 0.3859],
        [0.1446, 0.3859, 1.0000]])


### Embedding Generation Results

The cleaned and chunked robotics documents were converted into dense vector embeddings using the `sentence-transformers/all-MiniLM-L6-v2` model.

A total of **422 text chunks** were embedded successfully.

Each chunk was represented by a **384-dimensional vector**, producing an embedding matrix with shape:

**(422, 384)**

The embeddings were generated as `float32` values and normalized during encoding to support efficient cosine-similarity comparison.

The embedding pipeline was also validated by confirming that the number of generated embeddings matched the number of chunks:

- Number of chunks: **422**
- Number of embeddings: **422**
- One embedding per chunk: **Verified**

A semantic similarity test was performed using three example sentences. Two sentences describing Jacobians produced a cosine similarity score of approximately **0.6535**, while a Jacobian sentence compared with an unrelated mobile-robot sentence produced a much lower similarity score of approximately **0.1446**.

This demonstrates that the embedding model is capturing semantic meaning rather than relying only on exact keyword matching.

### Embedding Configuration

- Embedding model: **sentence-transformers/all-MiniLM-L6-v2**
- Number of embedded chunks: **422**
- Embedding dimensions: **384**
- Embedding data type: **float32**
- Normalization: **Enabled**
- Similarity approach: **Cosine similarity**
- Execution device: **CPU**

GPU acceleration was not required for this stage because the dataset is relatively small and the embedding process completed successfully on CPU.

##Step 2.3.11 — Add the Chroma section

### Vector Store — ChromaDB

The generated embeddings are stored in a persistent ChromaDB vector database.

For each chunk, the vector store contains:

- the chunk embedding,
- the original chunk text,
- the source PDF filename,
- the page number,
- the chunk index,
- and a unique chunk identifier.

The vector store is persisted to disk so that the FastAPI backend can load it directly without reprocessing the PDFs or regenerating embeddings for every application startup.

In [32]:
#Step 2.3.12 — Set the vector-store path
import chromadb

VECTOR_STORE_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"

VECTOR_STORE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Vector store directory:")
print(VECTOR_STORE_DIR)

Vector store directory:
c:\Users\yasser\Projects\rag-study-assistant\backend\data\vector_store


In [33]:
#Step 2.3.13 — Create the persistent Chroma client
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_DIR)
)

print("Persistent Chroma client created.")

Persistent Chroma client created.


In [34]:
#Step 2.3.14 — Create the collection
COLLECTION_NAME = "roborag_chunks"

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

print("Collection:", collection.name)

Collection: roborag_chunks


In [35]:
#Step 2.3.15 — Prepare everything Chroma needs
ids = [chunk["chunk_id"] for chunk in chunks]

documents_for_chroma = [
    chunk["text"]
    for chunk in chunks
]

metadatas = [
    {
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_index": chunk["chunk_index"]
    }
    for chunk in chunks
]

embeddings_for_chroma = embeddings.tolist()

print("IDs:", len(ids))
print("Documents:", len(documents_for_chroma))
print("Metadata records:", len(metadatas))
print("Embeddings:", len(embeddings_for_chroma))

IDs: 422
Documents: 422
Metadata records: 422
Embeddings: 422


In [36]:
#Step 2.3.16 — Store everything
collection.upsert(
    ids=ids,
    embeddings=embeddings_for_chroma,
    documents=documents_for_chroma,
    metadatas=metadatas
)

print("Chunks stored in ChromaDB.")

Chunks stored in ChromaDB.


In [37]:
print("Records in collection:", collection.count())

Records in collection: 422


In [38]:
#Step 2.3.17 — Inspect a stored record
sample = collection.get(
    ids=[ids[0]],
    include=["documents", "metadatas"]
)

print(sample)

{'ids': ['01_intro_to_robotics.pdf_p1_c0'], 'embeddings': None, 'documents': ['CSE 432 Robotics\nCourse Introduction\nAhmed Asker, Ph.D.\nAssociate Professor in Mechatronic and Robotics\nAhmed.asker@ejust.edu.eg\nEgypt-Japan University of Science and Technology\n1'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'source': '01_intro_to_robotics.pdf', 'page': 1, 'chunk_index': 0}]}


In [39]:
#Step 2.3.18 — Check that the files physically exist
vector_store_files = list(VECTOR_STORE_DIR.rglob("*"))

print("Vector store files:")
for file in vector_store_files[:20]:
    print(file)

Vector store files:
c:\Users\yasser\Projects\rag-study-assistant\backend\data\vector_store\4ecb6358-8972-4287-99d7-93d741c31c0a
c:\Users\yasser\Projects\rag-study-assistant\backend\data\vector_store\chroma.sqlite3
c:\Users\yasser\Projects\rag-study-assistant\backend\data\vector_store\4ecb6358-8972-4287-99d7-93d741c31c0a\data_level0.bin
c:\Users\yasser\Projects\rag-study-assistant\backend\data\vector_store\4ecb6358-8972-4287-99d7-93d741c31c0a\header.bin
c:\Users\yasser\Projects\rag-study-assistant\backend\data\vector_store\4ecb6358-8972-4287-99d7-93d741c31c0a\index_metadata.pickle
c:\Users\yasser\Projects\rag-study-assistant\backend\data\vector_store\4ecb6358-8972-4287-99d7-93d741c31c0a\length.bin
c:\Users\yasser\Projects\rag-study-assistant\backend\data\vector_store\4ecb6358-8972-4287-99d7-93d741c31c0a\link_lists.bin


In [40]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

COLLECTION_NAME = "roborag_chunks"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

TOP_K = 4
OLLAMA_MODEL = "llama3.2:latest"
MAX_RETRIEVAL_DISTANCE = 0.65

In [41]:
# Step 2.3.19 — Save final RAG configuration

import json
from pathlib import Path
import chromadb


# ---------------------------------------------------------
# Resolve project paths
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


DATA_DIR = PROJECT_ROOT / "data" / "raw"

VECTOR_STORE_DIR = (
    PROJECT_ROOT
    / "backend"
    / "data"
    / "vector_store"
)

CONFIG_PATH = (
    PROJECT_ROOT
    / "backend"
    / "data"
    / "rag_config.json"
)


# ---------------------------------------------------------
# Final configuration
# ---------------------------------------------------------

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

COLLECTION_NAME = "roborag_chunks"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

EMBEDDING_DIMENSION = 384

TOP_K = 4

OLLAMA_MODEL = "llama3.2:latest"

MAX_RETRIEVAL_DISTANCE = 0.65


# ---------------------------------------------------------
# Read persisted counts
# ---------------------------------------------------------

pdf_files = sorted(
    DATA_DIR.glob("*.pdf")
)

client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_DIR)
)

collection = client.get_collection(
    name=COLLECTION_NAME
)

number_of_documents = len(pdf_files)
number_of_chunks = collection.count()


# ---------------------------------------------------------
# Build config
# ---------------------------------------------------------

rag_config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "collection_name": COLLECTION_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "number_of_documents": number_of_documents,
    "number_of_chunks": number_of_chunks,

    # Final online RAG configuration
    "top_k": TOP_K,
    "ollama_model": OLLAMA_MODEL,
    "max_retrieval_distance": MAX_RETRIEVAL_DISTANCE,
}


# ---------------------------------------------------------
# Save config
# ---------------------------------------------------------

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        rag_config,
        f,
        indent=4,
    )


print(
    "Saved configuration to:",
    CONFIG_PATH
)

print(
    json.dumps(
        rag_config,
        indent=4
    )
)

Saved configuration to: c:\Users\yasser\Projects\rag-study-assistant\backend\data\rag_config.json
{
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "collection_name": "roborag_chunks",
    "chunk_size": 800,
    "chunk_overlap": 150,
    "embedding_dimension": 384,
    "number_of_documents": 11,
    "number_of_chunks": 422,
    "top_k": 4,
    "ollama_model": "llama3.2:latest",
    "max_retrieval_distance": 0.65
}


### Vector Store Results

The generated embeddings were successfully stored in a persistent **ChromaDB** vector database.

The vector store was created at:

`backend/data/vector_store/`

and contains a collection named:

`roborag_chunks`

A total of **422 records** were successfully stored in the collection, matching the total number of text chunks and generated embeddings.

Each record contains:

- a unique chunk ID,
- the original chunk text,
- a 384-dimensional embedding,
- source PDF filename,
- page number,
- and chunk index.

The stored data was verified by retrieving a sample record from the collection. The retrieved record correctly preserved both the original text and its metadata, including the source document and page number.

### Vector Store Configuration

- Vector database: **ChromaDB**
- Collection name: **roborag_chunks**
- Number of stored records: **422**
- Similarity metric: **Cosine distance**
- Embedding model: **sentence-transformers/all-MiniLM-L6-v2**
- Embedding dimensions: **384**
- Chunk size: **800 characters**
- Chunk overlap: **150 characters**
- Persistent storage: **Enabled**

The vector store was successfully written to disk and contains ChromaDB database files such as `chroma.sqlite3` and the associated vector index files.

A separate RAG configuration file was also saved at:

`backend/data/rag_config.json`

This configuration stores the embedding model name, collection name, chunk size, overlap, embedding dimension, number of source documents, and number of chunks.

Persisting both the vector store and configuration allows the future FastAPI backend to load the existing knowledge base directly without rereading the PDFs or regenerating the document embeddings at request time.

---

## Offline Indexing Phase Complete

The offline indexing stage of the RoboRAG pipeline is now complete.

During the offline phase, the source robotics documents were processed through the following steps:

1. **11 robotics lecture PDFs** were loaded and inspected.
2. A total of **418 pages** were successfully parsed.
3. Extracted text was lightly cleaned while preserving technical content.
4. The pages were divided into **422 page-aware text chunks**.
5. Each chunk was converted into a **384-dimensional embedding** using `sentence-transformers/all-MiniLM-L6-v2`.
6. The **422 embeddings**, original chunk texts, and source metadata were stored in a persistent **ChromaDB** vector database.
7. The vector store and RAG configuration were saved to disk so they can be reused without rebuilding the knowledge base.

The offline phase therefore produces the searchable knowledge base used by RoboRAG.

### Transition to the Online Query Phase

The project now moves to the **online RAG phase**.

Unlike the offline stage, which prepares the document knowledge base only once, the online stage is executed whenever a user submits a question.

The online pipeline is:

**User Question → Query Embedding → Vector Search → Top-K Relevant Chunks → Grounded Prompt → Ollama LLM → Answer + Sources**

The first step of the online phase is implementing and evaluating semantic retrieval from the persisted ChromaDB collection.

## Phase 2.4 — Retrieval & Prompting

## 2.4 Retrieval and Prompting

### Semantic Retrieval

During the online query phase, the user's question is converted into an embedding using the **same embedding model** that was used for the document chunks.

The query embedding is compared with the embeddings stored in ChromaDB using cosine distance.

The most semantically relevant chunks are returned as the **Top-K retrieved context**.

For RoboRAG, the initial retrieval configuration uses:

- Retrieval method: **Semantic vector search**
- Vector database: **ChromaDB**
- Embedding model: **sentence-transformers/all-MiniLM-L6-v2**
- Similarity metric: **Cosine**
- Top-K: **4**

A Top-K value of 4 was selected as an initial balance between retrieving enough supporting context and avoiding excessive or unrelated information in the final LLM prompt.

In [42]:
TOP_K = 4

print("Top-K retrieval:", TOP_K)

Top-K retrieval: 4


In [43]:
#step 2.4.3 — Embed a user question
def embed_query(question):
    """
    Convert a user question into the same embedding space
    used for the document chunks.
    """

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    return query_embedding[0].tolist()

In [44]:
test_question = "What is the Jacobian in robotics?"

test_query_embedding = embed_query(test_question)

print("Question:", test_question)
print("Query embedding dimensions:", len(test_query_embedding))

Question: What is the Jacobian in robotics?
Query embedding dimensions: 384


In [45]:
def normalize_query(question):
    """
    Make short or vague questions more explicit for
    robotics semantic retrieval.
    """

    question = " ".join(question.strip().split())

    lower_question = question.lower()

    definition_starters = (
        "what is",
        "what are",
        "define",
        "explain",
    )

    if lower_question.startswith(definition_starters):
        return (
            "Robotics course concept definition and explanation: "
            f"{question}"
        )

    return f"Robotics course question: {question}"

In [46]:
#Step 2.4.4 — Build the retrieval function
def retrieve_chunks(question, top_k=TOP_K):
    """
    Retrieve the most relevant document chunks
    for a user question.
    """

    retrieval_query = normalize_query(question)

    query_embedding = embed_query(
        retrieval_query
    )

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    retrieved_chunks = []

    for rank, (document, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ),
        start=1
    ):

        retrieved_chunks.append(
            {
                "rank": rank,
                "text": document,
                "source": metadata["source"],
                "page": metadata["page"],
                "chunk_index": metadata["chunk_index"],
                "distance": distance
            }
        )

    return retrieved_chunks

In [47]:
#Step 2.4.5 — Make results easy to inspect
def display_retrieval_results(question, top_k=TOP_K):

    results = retrieve_chunks(
        question,
        top_k=top_k
    )

    print("=" * 90)
    print("QUESTION:")
    print(question)
    print("=" * 90)

    for result in results:

        print()
        print(f"Rank: {result['rank']}")
        print(f"Source: {result['source']}")
        print(f"Page: {result['page']}")
        print(f"Distance: {result['distance']:.4f}")

        print("-" * 90)

        print(result["text"][:1000])

        print("=" * 90)

In [48]:
#Step 2.4.6 —  first real RoboRAG search
display_retrieval_results(
    "What is the Jacobian in robotics?"
)

QUESTION:
What is the Jacobian in robotics?

Rank: 1
Source: 07_velocity_kinematics.pdf
Page: 2
Distance: 0.3195
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
Introduction
• Forward kinematic equations define a function between the
space of Cartesian positions and orientations and the space of
joint positions.
• The velocity relationships are then determined by the Jacobian
of this function.
• The Jacobian is a matrix that can be thought of as the vector
version of the ordinary derivative of a scalar function.
• The Jacobian is one of the most important quantities in the
analysis and control of robot motion.
2CSE 432 Robotics

Rank: 2
Source: 08_jacobian.pdf
Page: 2
Distance: 0.3195
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
Introduction
• Forward kinematic equations define a function between the
space of Cartesian positions and orientations and the space of
join

In [49]:
display_retrieval_results(
    "What is a robot singularity?"
)

QUESTION:
What is a robot singularity?

Rank: 1
Source: 09_singularities.pdf
Page: 3
Distance: 0.4223
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
SINGULARITIES
• Singularities may correspond to workspace points that become
unreachable under small variations in link parameters (e.g.,
length, offsets).
• Near a singularity, the inverse kinematics may not have a
unique solution:
• There may be no solution.
• Or there may be infinitely many solutions.
CSE 432 Robotics 3

Rank: 2
Source: 01_intro_to_robotics.pdf
Page: 5
Distance: 0.4372
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
Introduction
• Robotics encompasses other areas not covered in this course
• locomotion, including wheeled and legged robots,
• flying and swimming robots,
• grasping,
• artificial intelligence and programming languages.
• This course is concerned with fundamentals of robotics,
including kin

In [50]:
display_retrieval_results(
    "What is forward kinematics?"
)

QUESTION:
What is forward kinematics?

Rank: 1
Source: 07_velocity_kinematics.pdf
Page: 2
Distance: 0.3385
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
Introduction
• Forward kinematic equations define a function between the
space of Cartesian positions and orientations and the space of
joint positions.
• The velocity relationships are then determined by the Jacobian
of this function.
• The Jacobian is a matrix that can be thought of as the vector
version of the ordinary derivative of a scalar function.
• The Jacobian is one of the most important quantities in the
analysis and control of robot motion.
2CSE 432 Robotics

Rank: 2
Source: 08_jacobian.pdf
Page: 2
Distance: 0.3385
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
Introduction
• Forward kinematic equations define a function between the
space of Cartesian positions and orientations and the space of
joint posi

In [51]:
display_retrieval_results(
    "What are the different types of mobile robot locomotion?"
)

QUESTION:
What are the different types of mobile robot locomotion?

Rank: 1
Source: 01_intro_to_robotics.pdf
Page: 5
Distance: 0.3302
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
Introduction
• Robotics encompasses other areas not covered in this course
• locomotion, including wheeled and legged robots,
• flying and swimming robots,
• grasping,
• artificial intelligence and programming languages.
• This course is concerned with fundamentals of robotics,
including kinematics, dynamics, motion planning , and
control.
6CSE 432 Robotics

Rank: 2
Source: 01_intro_to_robotics.pdf
Page: 9
Distance: 0.4384
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
What is a Robot?
• Robotics
• Scientific area that studies the link between Perception and Action
• Robot Arms
• Device able to perform activities as a human
• Programmable manipulator able to execute multiple operations,
fol

In [52]:
display_retrieval_results(
    "How is 3D rotation represented in robotics?"
)

QUESTION:
How is 3D rotation represented in robotics?

Rank: 1
Source: 02_rigid_motion_1.pdf
Page: 11
Distance: 0.2784
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
Rotations in three dimensions
• The projection technique described above scales nicely to the
3D case.
• Each axis of the frame 𝑜1𝑥1𝑦1𝑧1 is projected onto coordinate
frame 𝑜0𝑥0𝑦0𝑧0.
• The resulting rotation matrix is given by
𝑅1
0 =
𝑥1𝑥0 𝑦1𝑥0 𝑧1𝑥0
𝑥1𝑦0 𝑦1𝑦0 𝑧1𝑦0
𝑥1𝑧0 𝑦1𝑧0 𝑧1𝑧0
11CSE 432 Robotics

Rank: 2
Source: 02_rigid_motion_1.pdf
Page: 9
Distance: 0.3557
------------------------------------------------------------------------------------------
Dr. Ahmed Asker
Rotation in the plane
• A scalable 3D approach is to construct the rotation matrix by
projecting the axes of frame 𝑜1𝑥1𝑦1onto the axes of frame
𝑜0𝑥0𝑦0.
𝑅0
1 = 𝑥0𝑥1 𝑦0𝑥1
𝑥0𝑦1 𝑦0𝑦1
• Since dot product commutates
𝑅1
0 = 𝑥1𝑥0 𝑦1𝑥0
𝑥1𝑦0 𝑦1𝑦0
9CSE 432 Robotics

Rank: 3
Source: 02_rigid_motion_1.pdf
Page: 13
Dis

In [53]:
#Step 2.4.8 — Create a compact retrieval table
def retrieval_summary(question, top_k=TOP_K):

    results = retrieve_chunks(
        question,
        top_k=top_k
    )

    rows = []

    for result in results:

        rows.append(
            {
                "rank": result["rank"],
                "source": result["source"],
                "page": result["page"],
                "distance": round(result["distance"], 4),
                "preview": result["text"][:100]
            }
        )

    return pd.DataFrame(rows)

In [54]:
retrieval_summary(
    "What is a robot singularity?"
)

,rank,source,page,distance,preview
0,1,09_singularities.pdf,3,0.4223,Dr. Ahmed Asker\nSINGULARITIES\n• Singularitie...
1,2,01_intro_to_robotics.pdf,5,0.4372,Dr. Ahmed Asker\nIntroduction\n• Robotics enco...
2,3,01_intro_to_robotics.pdf,4,0.4624,Dr. Ahmed Asker\nIntroduction\n• Robotics is a...
3,4,01_intro_to_robotics.pdf,8,0.4750,Dr. Ahmed Asker\nIntroduction\n• Virtually any...


### Retrieval Validation Results

The semantic retrieval pipeline was tested using questions from several major robotics topics.

The retrieved results showed that the embedding model and ChromaDB vector search were able to locate relevant lecture content based on semantic meaning rather than exact filename or keyword matching.

Example retrieval results included:

- **"What is the Jacobian in robotics?"**
  - Rank 1: `07_velocity_kinematics.pdf`, Page 2
  - The retrieved slide directly explains that the Jacobian relates robot kinematic variables and describes it as an important matrix in robot motion analysis.

- **"What is a robot singularity?"**
  - Rank 1: `09_singularities.pdf`, Page 3
  - The retrieved content directly explains singularities and their effects on inverse-kinematic solutions.

- **"What is forward kinematics?"**
  - Rank 1: `07_velocity_kinematics.pdf`, Page 2
  - Although the result came from the velocity-kinematics lecture, the retrieved slide contains a relevant explanation of forward-kinematic equations.

- **"What are the different types of mobile robot locomotion?"**
  - Rank 1: `10_mobile_robots_1.pdf`, Page 16
  - The retrieved content lists several locomotion methods including walking, jumping, running, sliding, skating, swimming, flying, and rolling.

- **"How is 3D rotation represented in robotics?"**
  - Rank 1: `02_rigid_motion_1.pdf`, Page 11
  - The retrieved slide directly discusses three-dimensional rotations and the corresponding rotation matrix.

These tests indicate that the semantic retrieval pipeline is successfully retrieving relevant context from the robotics knowledge base.

The retrieval experiments confirmed that the embedding model and
ChromaDB consistently locate relevant robotics material.

The final RoboRAG application uses **Top-K = 4** to provide the
generation model with a broader set of relevant course context.

### Grounded Prompt Construction

After retrieving the Top-K relevant chunks, RoboRAG combines the retrieved context with the user's question to construct an augmented prompt for the local LLM.

The prompt instructs the model to:

- answer using only the retrieved robotics course material,
- avoid relying on unsupported external knowledge,
- state when the available context does not contain the answer,
- and preserve source information so that the final response can be associated with the original lecture documents.

This grounding mechanism reduces hallucination and ensures that the generated answer is supported by the RoboRAG knowledge base.

In [55]:
#Step 2.4.10 — Build the context
def build_context(retrieved_chunks):
    """
    Combine retrieved chunks into a formatted context string.
    """

    context_parts = []

    for result in retrieved_chunks:
        context_part = (
            f"[Source: {result['source']}, Page: {result['page']}]\n"
            f"{result['text']}"
        )

        context_parts.append(context_part)

    return "\n\n".join(context_parts)

In [56]:
retrieved = retrieve_chunks(
    "What is the Jacobian in robotics?"
)

context = build_context(retrieved)

print(context)

[Source: 07_velocity_kinematics.pdf, Page: 2]
Dr. Ahmed Asker
Introduction
• Forward kinematic equations define a function between the
space of Cartesian positions and orientations and the space of
joint positions.
• The velocity relationships are then determined by the Jacobian
of this function.
• The Jacobian is a matrix that can be thought of as the vector
version of the ordinary derivative of a scalar function.
• The Jacobian is one of the most important quantities in the
analysis and control of robot motion.
2CSE 432 Robotics

[Source: 08_jacobian.pdf, Page: 2]
Dr. Ahmed Asker
Introduction
• Forward kinematic equations define a function between the
space of Cartesian positions and orientations and the space of
joint positions.
• The velocity relationships are then determined by the Jacobian
of this function.
• The Jacobian is a matrix that can be thought of as the vector
version of the ordinary derivative of a scalar function.
• The Jacobian is one of the most important quantities

In [57]:
#Step 2.4.11 — Build the grounded prompt
def build_prompt(question, retrieved_chunks):
    context = build_context(retrieved_chunks)

    prompt = f"""
You are RoboRAG, a robotics course assistant.

Use the retrieved robotics course context below to answer the question.

If the context contains information that answers the question,
give a short technical answer based only on that information.

Only say:
"I could not find this information in the provided robotics course documents."
when the retrieved context truly does not contain enough information.

Do not use outside knowledge.
Do not invent information.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    return prompt.strip()

In [58]:
question = "What is a robot singularity?"

retrieved = retrieve_chunks(question)

prompt = build_prompt(
    question,
    retrieved
)

print(prompt)

You are RoboRAG, a robotics course assistant.

Use the retrieved robotics course context below to answer the question.

If the context contains information that answers the question,
give a short technical answer based only on that information.

Only say:
"I could not find this information in the provided robotics course documents."
when the retrieved context truly does not contain enough information.

Do not use outside knowledge.
Do not invent information.

CONTEXT:
[Source: 09_singularities.pdf, Page: 3]
Dr. Ahmed Asker
SINGULARITIES
• Singularities may correspond to workspace points that become
unreachable under small variations in link parameters (e.g.,
length, offsets).
• Near a singularity, the inverse kinematics may not have a
unique solution:
• There may be no solution.
• Or there may be infinitely many solutions.
CSE 432 Robotics 3

[Source: 01_intro_to_robotics.pdf, Page: 5]
Dr. Ahmed Asker
Introduction
• Robotics encompasses other areas not covered in this course
• locomoti

### Local LLM Generation with Ollama

The retrieved context and user question are passed to a local Large Language Model using Ollama.

RoboRAG uses `llama3.2:latest` for local answer generation. A lightweight model was selected because it can run efficiently on the available hardware while remaining suitable for grounded question answering.

The model does not receive the complete robotics document collection. Instead, it receives only the Top-K chunks retrieved for the current question.

This creates the complete online RAG flow:

**Question → Query Embedding → Retrieval → Context Construction → Grounded Prompt → Ollama → Answer**

A low generation temperature is used to encourage more deterministic and grounded responses.

In [59]:
import ollama

OLLAMA_MODEL = "llama3.2:latest"

print("Ollama model:", OLLAMA_MODEL)

Ollama model: llama3.2:latest


In [60]:
#Step 2.4.13 — Generation function
FALLBACK_RESPONSE = (
    "I could not find this information in the provided "
    "robotics course documents."
)


def generate_answer(question, retrieved_chunks):
    """
    Generate a useful but strictly grounded study answer.
    """

    context = build_context(retrieved_chunks)

    system_message = """
You are RoboRAG, a university robotics course assistant.

Your ONLY factual source is the retrieved robotics course context.

Your goal is to explain the concept clearly enough for a student
to study it, while remaining strictly grounded in the retrieved material.

ANSWER STYLE

For conceptual questions:

1. Start with a direct definition or answer.

2. Then explain additional relevant information found in the context.

3. When supported by the context, explain:
   - how the concept relates to another concept,
   - why it is important,
   - its properties,
   - or its consequences.

4. Normally write 3-6 informative sentences.

5. Use two short paragraphs or bullet points when useful.

6. Do not repeat the same definition several times using different wording.

STRICT GROUNDING

7. Every technical statement must be supported by the retrieved context.

8. You may paraphrase and combine facts from multiple retrieved chunks.

9. Do NOT add examples unless the examples explicitly appear in the context.

10. Do NOT add applications, tasks, robot components, equations,
    formulas, or terminology unless explicitly supported by the context.

11. Do NOT use general pretrained robotics knowledge to make
    the answer sound more complete.

12. Never invent information merely to make the answer longer.

13. If the retrieved material supports only a short explanation,
    give a short explanation.

14. Do not begin with fragments such as "The Jacobian."
    Begin with a complete sentence.

15. If the context genuinely does not contain enough information,
    reply exactly:

"I could not find this information in the provided robotics course documents."

16. If the retrieved context clearly contains the answer,
    do not refuse.
"""

    user_message = f"""
QUESTION:
{question}

RETRIEVED ROBOTICS COURSE CONTEXT:
{context}

Write a clear study explanation using only information supported
by the retrieved context.
"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_message,
            },
            {
                "role": "user",
                "content": user_message,
            },
        ],
        options={
            "temperature": 0,
            "num_predict": 350,
        },
    )

    answer = response["message"]["content"].strip()

    # Retry only when the model refuses despite strong retrieval.
    if (
        answer == FALLBACK_RESPONSE
        and retrieved_chunks
        and retrieved_chunks[0]["distance"] <= 0.50
    ):

        retry_message = f"""
The retrieved context is strongly relevant to the student's question.

QUESTION:
{question}

CONTEXT:
{context}

Your previous response incorrectly refused to answer.

Read the context carefully and answer using ONLY facts supported
by the retrieved course material.

Give a clear explanation of about 3-6 sentences if the context
contains enough information.

Do not add outside knowledge, examples, applications, or equations.
"""

        retry_response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": system_message,
                },
                {
                    "role": "user",
                    "content": retry_message,
                },
            ],
            options={
                "temperature": 0,
                "num_predict": 350,
            },
        )

        answer = retry_response["message"]["content"].strip()

    return answer

In [61]:
# Step 2.4.14 — First full generated answer

question = "What is a robot singularity?"

retrieved = retrieve_chunks(
    question,
    top_k=TOP_K
)

answer = generate_answer(
    question,
    retrieved
)

print(answer)

A robot singularity refers to a workspace point that becomes unreachable under small variations in link parameters, such as length or offsets.

In the context of robotics, a singularity can occur due to the limitations of the robot's kinematic structure, making it difficult or impossible to achieve a specific pose or configuration. This can lead to a situation where the inverse kinematics problem has no unique solution, resulting in either no solution or infinitely many solutions.


In [62]:
#Step 2.4.15 — Build source citations
def extract_sources(retrieved_chunks):
    """
    Return unique source/page citations from retrieved chunks.
    """

    sources = []

    for result in retrieved_chunks:
        source = f"{result['source']} - Page {result['page']}"

        if source not in sources:
            sources.append(source)

    return sources

In [63]:
sources = extract_sources(retrieved)

for source in sources:
    print(source)

09_singularities.pdf - Page 3
01_intro_to_robotics.pdf - Page 5
01_intro_to_robotics.pdf - Page 4
01_intro_to_robotics.pdf - Page 8


In [64]:
def rag_answer(question, top_k=TOP_K):
    """
    Complete RoboRAG online query pipeline.
    """

    retrieved = retrieve_chunks(
        question,
        top_k=top_k
    )

    # Reject clearly unrelated questions
    if (
        not retrieved
        or retrieved[0]["distance"]
        > MAX_RETRIEVAL_DISTANCE
    ):
        return {
            "question": question,
            "answer": (
                "I could not find this information in the "
                "provided robotics course documents."
            ),
            "sources": [],
            "retrieved_chunks": retrieved,
        }

    answer = generate_answer(
        question,
        retrieved
    )

    sources = extract_sources(
        retrieved
    )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": retrieved,
    }

In [65]:
#Step 2.4.17 — Test the full pipeline
result = rag_answer(
    "What is the Jacobian in robotics?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print("-", source)

QUESTION:
What is the Jacobian in robotics?

ANSWER:
The Jacobian in robotics is a matrix that represents the vector version of the ordinary derivative of a scalar function.

The Jacobian is a crucial quantity in the analysis and control of robot motion, as it determines the velocity relationships between the space of joint positions and the space of Cartesian positions and orientations. In the context of forward kinematic equations, the Jacobian is used to find the velocity relationships between the joint positions and the desired Cartesian positions and orientations.

SOURCES:
- 07_velocity_kinematics.pdf - Page 2
- 08_jacobian.pdf - Page 2
- 01_intro_to_robotics.pdf - Page 5
- 08_jacobian.pdf - Page 32


In [66]:
#Step 2.4.18 — Critical hallucination test
result = rag_answer(
    "What was the revenue of Microsoft in 2025?"
)

print(result["answer"])

I could not find this information in the provided robotics course documents.


In [67]:
question = "What is the Jacobian in robotics?"

retrieved = retrieve_chunks(
    question,
    top_k=1
)

answer = generate_answer(
    question,
    retrieved
)

print(answer)

The Jacobian in robotics is a matrix that represents the relationship between the velocity of a robot's end effector and the velocity of its joint angles.

The Jacobian is a crucial concept in robotics, as it allows us to determine the velocity relationships between the joint positions and the desired Cartesian positions and orientations. In the context of the retrieved material, the Jacobian is described as the "vector version of the ordinary derivative of a scalar function." This means that the Jacobian matrix can be thought of as a way to compute the partial derivatives of a function with respect to multiple variables, in this case, the joint angles.


In [68]:
result = rag_answer(
    "What is the Jacobian in robotics?",
    top_k=4
)

print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print("-", source)

The Jacobian in robotics is a matrix that represents the vector version of the ordinary derivative of a scalar function.

The Jacobian is a crucial quantity in the analysis and control of robot motion, as it determines the velocity relationships between the space of Cartesian positions and orientations and the space of joint positions. This matrix is derived from the forward kinematic equations, which define a function between the space of Cartesian positions and orientations and the space of joint positions.

Sources:
- 07_velocity_kinematics.pdf - Page 2
- 08_jacobian.pdf - Page 2
- 01_intro_to_robotics.pdf - Page 5
- 08_jacobian.pdf - Page 32


In [69]:
result = rag_answer(
    "What is a robot singularity?",
    top_k=4
)

print(result["answer"])

A robot singularity refers to a workspace point that becomes unreachable under small variations in link parameters, such as length or offsets.

In the context of robotics, a singularity can occur due to the limitations of the robot's kinematic structure, making it difficult or impossible to achieve a specific pose or configuration. This can lead to a situation where the inverse kinematics problem has no unique solution, resulting in either no solution or infinitely many solutions.


In [70]:
result = rag_answer(
    "What was Microsoft's revenue in 2025?",
    top_k=4
)

print(result["answer"])

I could not find this information in the provided robotics course documents.


In [71]:
result = rag_answer(
    "What is a robot singularity?",
    top_k=4
)

print(result["answer"])

A robot singularity refers to a workspace point that becomes unreachable under small variations in link parameters, such as length or offsets.

In the context of robotics, a singularity can occur due to the limitations of the robot's kinematic structure, making it difficult or impossible to achieve certain positions or orientations. This can lead to a situation where the inverse kinematics problem has no unique solution, resulting in either no solution or infinitely many solutions.


## 2.6 RAG Pipeline Evaluation

The completed RoboRAG pipeline is evaluated using a set of questions covering different robotics topics represented in the document collection.

The evaluation examines two major parts of the RAG pipeline:

**Retrieval Quality:**  
Whether the retrieved document chunks contain information relevant to the user's question.

**Generation Quality:**  
Whether the generated answer is correct and supported by the retrieved context without introducing unsupported information.

The evaluation also includes questions whose answers are intentionally not present in the robotics knowledge base. These questions are used to test whether RoboRAG can correctly refuse to answer rather than hallucinating information.

For each question, the following information is recorded:

- retrieved source document and page,
- generated answer,
- retrieval relevance,
- answer correctness,
- grounding / hallucination behavior.

A total of **12 questions** are used in the evaluation.

In [72]:
print("TOP_K:", TOP_K)
print("OLLAMA_MODEL:", OLLAMA_MODEL)
print("MAX_RETRIEVAL_DISTANCE:", MAX_RETRIEVAL_DISTANCE)
print("Stored chunks:", collection.count())

TOP_K: 4
OLLAMA_MODEL: llama3.2:latest
MAX_RETRIEVAL_DISTANCE: 0.65
Stored chunks: 422


In [73]:
sanity_questions = [
    "What is the Jacobian in robotics?",
    "What is forward kinematics?",
    "What was Microsoft's revenue in 2025?"
]

for question in sanity_questions:
    result = rag_answer(question)

    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")
    for source in result["sources"]:
        print("-", source)

    print()

QUESTION:
What is the Jacobian in robotics?

ANSWER:
The Jacobian in robotics is a matrix that represents the vector version of the ordinary derivative of a scalar function.

The Jacobian is a crucial quantity in the analysis and control of robot motion, as it determines the velocity relationships between the space of Cartesian positions and orientations and the space of joint positions. This matrix is derived from the forward kinematic equations, which define a function between the space of Cartesian positions and orientations and the space of joint positions.

SOURCES:
- 07_velocity_kinematics.pdf - Page 2
- 08_jacobian.pdf - Page 2
- 01_intro_to_robotics.pdf - Page 5
- 08_jacobian.pdf - Page 32

QUESTION:
What is forward kinematics?

ANSWER:
Forward kinematics is a function that defines the relationship between the space of Cartesian positions and orientations and the space of joint positions.

This function is crucial in robotics as it allows us to determine the position and orient

In [74]:
test_questions = [
    "What is forward kinematics?",
    "What is the Jacobian in robotics?",
    "What happens near a robot singularity?",
    "What was Microsoft's revenue in 2025?"
]

for question in test_questions:

    result = rag_answer(question)

    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(result["answer"])

    print("\nWORD COUNT:")
    print(len(result["answer"].split()))

    print("\nSOURCES:")
    for source in result["sources"]:
        print("-", source)

    print()

QUESTION:
What is forward kinematics?

ANSWER:
Forward kinematics is a function that defines the relationship between the space of Cartesian positions and orientations and the space of joint positions.

This function is crucial in robotics as it allows us to determine the position and orientation of a robot's end effector based on the joint positions. In other words, forward kinematics enables us to calculate the robot's pose in the Cartesian space given the joint angles.

WORD COUNT:
69

SOURCES:
- 07_velocity_kinematics.pdf - Page 2
- 08_jacobian.pdf - Page 2
- 01_intro_to_robotics.pdf - Page 5
- 05_forward_kinematics_1.pdf - Page 29

QUESTION:
What is the Jacobian in robotics?

ANSWER:
The Jacobian in robotics is a matrix that represents the vector version of the ordinary derivative of a scalar function.

The Jacobian is a crucial quantity in the analysis and control of robot motion, as it determines the velocity relationships between the space of Cartesian positions and orientation

In [75]:
#Step 2.6.1 — Create the evaluation questions
evaluation_questions = [
    # Robotics questions
    "What is the Jacobian in robotics?",
    "What is a robot singularity?",
    "What is forward kinematics?",
    "What is a rotation matrix?",
    "How are rotations represented in three dimensions?",
    "What is velocity kinematics?",
    "What is the relationship between joint velocities and robot motion?",
    "What are the different types of mobile robot locomotion?",
    "Why is the Jacobian important in robot motion?",
    "What problems can occur near a robot singularity?",

    # Unsupported questions
    "What was Microsoft's revenue in 2025?",
    "What is the capital city of Australia?"
]

print("Number of evaluation questions:", len(evaluation_questions))

Number of evaluation questions: 12


In [76]:
#Step 2.6.2 — Run RoboRAG on all 12
evaluation_results = []

for i, question in enumerate(evaluation_questions, start=1):

    print(f"Running question {i}/{len(evaluation_questions)}...")

    result = rag_answer(
        question,
        top_k=4
    )

    top_result = result["retrieved_chunks"][0]

    evaluation_results.append(
        {
            "question": question,
            "retrieved_source": top_result["source"],
            "retrieved_page": top_result["page"],
            "retrieval_distance": round(top_result["distance"], 4),
            "answer": result["answer"]
        }
    )

print("\nEvaluation complete.")

Running question 1/12...
Running question 2/12...
Running question 3/12...
Running question 4/12...
Running question 5/12...
Running question 6/12...
Running question 7/12...
Running question 8/12...
Running question 9/12...
Running question 10/12...
Running question 11/12...
Running question 12/12...

Evaluation complete.


In [77]:
#Step 2.6.3 — Create the results table
evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,question,retrieved_source,retrieved_page,retrieval_distance,answer
0,What is the Jacobian in robotics?,07_velocity_kinematics.pdf,2,0.3195,The Jacobian in robotics is a matrix that repr...
1,What is a robot singularity?,09_singularities.pdf,3,0.4223,A robot singularity refers to a workspace poin...
2,What is forward kinematics?,07_velocity_kinematics.pdf,2,0.3385,Forward kinematics is a function that defines ...
3,What is a rotation matrix?,02_rigid_motion_1.pdf,25,0.3306,A rotation matrix is a mathematical representa...
4,How are rotations represented in three dimensi...,02_rigid_motion_1.pdf,11,0.2265,Rotations in three dimensions are represented ...
5,What is velocity kinematics?,07_velocity_kinematics.pdf,2,0.3775,Velocity kinematics refers to the relationship...
6,What is the relationship between joint velocit...,07_velocity_kinematics.pdf,2,0.3785,The relationship between joint velocities and ...
7,What are the different types of mobile robot l...,01_intro_to_robotics.pdf,5,0.3302,"According to the retrieved context, there are ..."
8,Why is the Jacobian important in robot motion?,07_velocity_kinematics.pdf,2,0.2749,The Jacobian is important in robot motion beca...
9,What problems can occur near a robot singularity?,09_singularities.pdf,3,0.3440,A robot singularity can occur when the workspa...


In [78]:
#Step 2.6.4 — Add the manual evaluation columns
evaluation_df["retrieval_relevant"] = ""
evaluation_df["answer_correct"] = ""
evaluation_df["grounded"] = ""
evaluation_df["notes"] = ""

evaluation_df

,question,retrieved_source,retrieved_page,retrieval_distance,answer,retrieval_relevant,answer_correct,grounded,notes
0,What is the Jacobian in robotics?,07_velocity_kinematics.pdf,2,0.3195,The Jacobian in robotics is a matrix that repr...,,,,
1,What is a robot singularity?,09_singularities.pdf,3,0.4223,A robot singularity refers to a workspace poin...,,,,
2,What is forward kinematics?,07_velocity_kinematics.pdf,2,0.3385,Forward kinematics is a function that defines ...,,,,
3,What is a rotation matrix?,02_rigid_motion_1.pdf,25,0.3306,A rotation matrix is a mathematical representa...,,,,
4,How are rotations represented in three dimensi...,02_rigid_motion_1.pdf,11,0.2265,Rotations in three dimensions are represented ...,,,,
5,What is velocity kinematics?,07_velocity_kinematics.pdf,2,0.3775,Velocity kinematics refers to the relationship...,,,,
6,What is the relationship between joint velocit...,07_velocity_kinematics.pdf,2,0.3785,The relationship between joint velocities and ...,,,,
7,What are the different types of mobile robot l...,01_intro_to_robotics.pdf,5,0.3302,"According to the retrieved context, there are ...",,,,
8,Why is the Jacobian important in robot motion?,07_velocity_kinematics.pdf,2,0.2749,The Jacobian is important in robot motion beca...,,,,
9,What problems can occur near a robot singularity?,09_singularities.pdf,3,0.3440,A robot singularity can occur when the workspa...,,,,


In [79]:
#Step 2.6.5 — Make inspection easier
def inspect_evaluation_result(index):
    row = evaluation_results[index]

    print("=" * 90)
    print("QUESTION:")
    print(row["question"])

    print("\nTOP RETRIEVED SOURCE:")
    print(
        f"{row['retrieved_source']} "
        f"- Page {row['retrieved_page']}"
    )

    print("\nDISTANCE:")
    print(row["retrieval_distance"])

    print("\nANSWER:")
    print(row["answer"])

    print("=" * 90)

In [80]:
inspect_evaluation_result(0)

QUESTION:
What is the Jacobian in robotics?

TOP RETRIEVED SOURCE:
07_velocity_kinematics.pdf - Page 2

DISTANCE:
0.3195

ANSWER:
The Jacobian in robotics is a matrix that represents the vector version of the ordinary derivative of a scalar function.

The Jacobian is a crucial quantity in the analysis and control of robot motion, as it determines the velocity relationships between the space of joint positions and the space of Cartesian positions and orientations. In the context of forward kinematic equations, the Jacobian is used to find the velocity relationships between the joint positions and the desired Cartesian positions and orientations.


In [81]:
for i, row in evaluation_df.iterrows():
    print("=" * 100)
    print(f"QUESTION {i}")
    print(row["question"])

    print("\nSOURCE:")
    print(
        row["retrieved_source"],
        "- Page",
        row["retrieved_page"]
    )

    print("\nDISTANCE:")
    print(row["retrieval_distance"])

    print("\nFULL ANSWER:")
    print(row["answer"])
    print()

QUESTION 0
What is the Jacobian in robotics?

SOURCE:
07_velocity_kinematics.pdf - Page 2

DISTANCE:
0.3195

FULL ANSWER:
The Jacobian in robotics is a matrix that represents the vector version of the ordinary derivative of a scalar function.

The Jacobian is a crucial quantity in the analysis and control of robot motion, as it determines the velocity relationships between the space of joint positions and the space of Cartesian positions and orientations. In the context of forward kinematic equations, the Jacobian is used to find the velocity relationships between the joint positions and the desired Cartesian positions and orientations.

QUESTION 1
What is a robot singularity?

SOURCE:
09_singularities.pdf - Page 3

DISTANCE:
0.4223

FULL ANSWER:
A robot singularity refers to a workspace point that becomes unreachable under small variations in link parameters, such as length or offsets.

In the context of robotics, a singularity can occur due to the limitations of the robot's kinematic

In [82]:
evaluation_df.loc[0, [
    "retrieval_relevant",
    "answer_correct",
    "grounded",
    "notes"
]] = [
    "Yes",
    "Yes",
    "Yes",
    "Correct Jacobian definition retrieved and answered."
]

In [83]:
def inspect_question(index):
    row = evaluation_df.iloc[index]
    question = row["question"]

    retrieved = retrieve_chunks(question, top_k=4)

    print("=" * 100)
    print(f"QUESTION {index}:")
    print(question)

    print("\nGENERATED ANSWER:")
    print(row["answer"])

    print("\nRETRIEVED CONTEXT:")

    for result in retrieved:
        print("-" * 100)
        print(
            f"Rank {result['rank']} | "
            f"{result['source']} | "
            f"Page {result['page']} | "
            f"Distance {result['distance']:.4f}"
        )
        print(result["text"])

    print("=" * 100)

In [84]:
inspect_question(2)

QUESTION 2:
What is forward kinematics?

GENERATED ANSWER:
Forward kinematics is a function that defines the relationship between the space of Cartesian positions and orientations and the space of joint positions.

This function is crucial in robotics as it allows us to determine the position and orientation of a robot's end effector based on the joint positions. In other words, forward kinematics enables us to calculate the robot's pose in the Cartesian space given the joint angles.

RETRIEVED CONTEXT:
----------------------------------------------------------------------------------------------------
Rank 1 | 07_velocity_kinematics.pdf | Page 2 | Distance 0.3385
Dr. Ahmed Asker
Introduction
• Forward kinematic equations define a function between the
space of Cartesian positions and orientations and the space of
joint positions.
• The velocity relationships are then determined by the Jacobian
of this function.
• The Jacobian is a matrix that can be thought of as the vector
version of 

In [85]:
inspect_question(3)

QUESTION 3:
What is a rotation matrix?

GENERATED ANSWER:
A rotation matrix is a mathematical representation of a rotation in 3D space.

It can be described as a product of successive rotations about the principal coordinate axes x0, y0, and z0 "Fixed Frame" taken in a specific order. These rotations define the roll, pitch, and yaw angles, which are denoted as θ, φ, and ψ respectively. The rotation order is z → y → x.

The rotation matrix can also be represented as a product of individual rotation matrices about each axis, in a specific order.

RETRIEVED CONTEXT:
----------------------------------------------------------------------------------------------------
Rank 1 | 02_rigid_motion_1.pdf | Page 25 | Distance 0.3306
Dr. Ahmed Asker
Example 7
• Suppose that a rotation matrix R represents a rotation of angle
𝜙 about 𝑦0 followed by a rotation of angle 𝜃 about the fixed 𝑧0.
𝑅 = 𝑅𝑧,𝜃𝑅𝑦,𝜙
𝑅 =
cos 𝜙 cos 𝜃 − sin 𝜃 sin 𝜙 cos 𝜃
cos 𝜙 sin 𝜃 cos 𝜃 sin 𝜙 sin 𝜃
− sin 𝜙 0 cos 𝜙
25CSE 432 Robotics

In [86]:
inspect_question(4)

QUESTION 4:
How are rotations represented in three dimensions?

GENERATED ANSWER:
Rotations in three dimensions are represented by a rotation matrix, which is a 3x3 matrix. The rotation matrix is given by:

𝑅1
0 =
𝑥1𝑥0 𝑦1𝑥0 𝑧1𝑥0
𝑥1𝑦0 𝑦1𝑦0 𝑧1𝑦0
𝑥1𝑧0 𝑦1𝑧0 𝑧1𝑧0

This matrix represents a rotation about the principal axes of the frame 𝑜1𝑥1𝑦1, which are projected onto the coordinate frame 𝑜0𝑥0𝑦0.

The rotation matrix has several important properties:

* The transpose of the rotation matrix is equal to its inverse, i.e. 𝑅−1 = 𝑅𝑇.
* The columns (and therefore the rows) of the rotation matrix are mutually orthogonal.
* Each column (and therefore each row) of the rotation matrix is a unit vector.
* The determinant of the rotation matrix is equal to 1.

These properties apply to rotation matrices in three or more dimensions.

RETRIEVED CONTEXT:
----------------------------------------------------------------------------------------------------
Rank 1 | 02_rigid_motion_1.pdf | Page 11 | Distance 0

In [87]:
inspect_question(5)

QUESTION 5:
What is velocity kinematics?

GENERATED ANSWER:
Velocity kinematics refers to the relationship between the velocity of a robot's end effector and the joint velocities of the robot.

The velocity relationships are determined by the Jacobian of the forward kinematic equations, which define a function between the space of Cartesian positions and orientations and the space of joint positions. The Jacobian is a matrix that represents the vector version of the ordinary derivative of a scalar function, and it plays a crucial role in the analysis and control of robot motion.

RETRIEVED CONTEXT:
----------------------------------------------------------------------------------------------------
Rank 1 | 07_velocity_kinematics.pdf | Page 2 | Distance 0.3775
Dr. Ahmed Asker
Introduction
• Forward kinematic equations define a function between the
space of Cartesian positions and orientations and the space of
joint positions.
• The velocity relationships are then determined by the Jacob

In [88]:
inspect_question(6)

QUESTION 6:
What is the relationship between joint velocities and robot motion?

GENERATED ANSWER:
The relationship between joint velocities and robot motion is described by the Jacobian of the forward kinematic equations.

The Jacobian is a matrix that represents the vector version of the ordinary derivative of a scalar function. In the context of robot motion, the Jacobian is used to determine the velocity relationships between the joint positions and the Cartesian positions and orientations.

The Jacobian is crucial in the analysis and control of robot motion, as it provides a way to relate the joint velocities to the desired robot motion. By understanding the Jacobian, one can better control the robot's motion and avoid potential issues such as the robot crashing into objects or oscillating about its desired position.

In essence, the Jacobian acts as a bridge between the joint velocities and the robot's motion, allowing for more precise control and manipulation of the robot's move

In [89]:
inspect_question(7)

QUESTION 7:
What are the different types of mobile robot locomotion?

GENERATED ANSWER:
According to the retrieved context, there are several types of mobile robot locomotion. These include:

* Walking
* Jumping
* Running
* Sliding
* Skating
* Swimming
* Flying
* Rolling

These locomotion methods are inspired by their biological counterparts and have been demonstrated in research robots.

RETRIEVED CONTEXT:
----------------------------------------------------------------------------------------------------
Rank 1 | 01_intro_to_robotics.pdf | Page 5 | Distance 0.3302
Dr. Ahmed Asker
Introduction
• Robotics encompasses other areas not covered in this course
• locomotion, including wheeled and legged robots,
• flying and swimming robots,
• grasping,
• artificial intelligence and programming languages.
• This course is concerned with fundamentals of robotics,
including kinematics, dynamics, motion planning , and
control.
6CSE 432 Robotics
---------------------------------------------------

In [90]:
inspect_question(8)

QUESTION 8:
Why is the Jacobian important in robot motion?

GENERATED ANSWER:
The Jacobian is important in robot motion because it determines the velocity relationships between the joint positions and the Cartesian positions and orientations.

The Jacobian is a matrix that represents the vector version of the ordinary derivative of a scalar function. In the context of robot motion, the Jacobian is used to relate the changes in joint positions to the changes in Cartesian positions and orientations. This is crucial for understanding how the robot's motion is affected by the joint positions.

The importance of the Jacobian lies in its ability to provide a mathematical representation of the robot's motion. By analyzing the Jacobian, one can gain insights into the robot's velocity relationships, which is essential for controlling and analyzing robot motion.

RETRIEVED CONTEXT:
----------------------------------------------------------------------------------------------------
Rank 1 | 07_ve

In [91]:
inspect_question(9)

QUESTION 9:
What problems can occur near a robot singularity?

GENERATED ANSWER:
A robot singularity can occur when the workspace point becomes unreachable under small variations in link parameters, such as length or offsets. This can lead to problems in the robot's performance and control.

Near a singularity, the inverse kinematics may not have a unique solution, resulting in either no solution or infinitely many solutions. This is because at a singularity, bounded end-effector velocities may require unbounded joint velocities, and bounded end-effector forces/torques may require unbounded joint torques.

RETRIEVED CONTEXT:
----------------------------------------------------------------------------------------------------
Rank 1 | 09_singularities.pdf | Page 3 | Distance 0.3440
Dr. Ahmed Asker
SINGULARITIES
• Singularities may correspond to workspace points that become
unreachable under small variations in link parameters (e.g.,
length, offsets).
• Near a singularity, the inverse kin

In [92]:
manual_evaluation = [
    # retrieval_relevant, answer_correct, grounded, notes
    ("Yes", "Yes", "Yes",
     "Correct Jacobian definition supported directly by the retrieved context."),

    ("Yes", "Yes", "Mostly",
     "Main concept is correct, but the answer introduced minor explanatory wording not explicitly present in the retrieved context."),

    ("Yes", "Yes", "Yes",
     "Correct definition of forward kinematics based directly on the retrieved lecture content."),

    ("Yes", "Yes", "Yes",
     "Correct explanation of a rotation matrix based on the retrieved rigid-motion material."),

    ("Yes", "Yes", "Yes",
     "Correctly identifies the rotation matrix as the representation used for three-dimensional rotations."),

    ("Yes", "Yes", "Yes",
     "Correctly explains that velocity relationships are determined using the Jacobian of the forward-kinematic function."),

    ("Yes", "Yes", "Yes",
     "Correctly relates joint velocities and robot motion through the Jacobian."),

    ("Yes", "Yes", "Yes",
     "Correctly lists the locomotion methods explicitly contained in the retrieved mobile-robot lecture."),

    ("Yes", "No", "Yes",
     "Relevant context was retrieved and explicitly contained the answer, but the LLM incorrectly refused to answer."),

    ("Yes", "Yes", "Yes",
     "Correctly identifies the inverse-kinematics problems that can occur near a singularity."),

    ("N/A", "Yes", "Yes",
     "Unsupported question was correctly rejected instead of being answered from pretrained knowledge."),

    ("N/A", "Yes", "Yes",
     "Unsupported question was correctly rejected instead of being answered from pretrained knowledge.")
]

for i, values in enumerate(manual_evaluation):
    evaluation_df.loc[i, [
        "retrieval_relevant",
        "answer_correct",
        "grounded",
        "notes"
    ]] = values

evaluation_df

,question,retrieved_source,retrieved_page,retrieval_distance,answer,retrieval_relevant,answer_correct,grounded,notes
0,What is the Jacobian in robotics?,07_velocity_kinematics.pdf,2,0.3195,The Jacobian in robotics is a matrix that repr...,Yes,Yes,Yes,Correct Jacobian definition supported directly...
1,What is a robot singularity?,09_singularities.pdf,3,0.4223,A robot singularity refers to a workspace poin...,Yes,Yes,Mostly,"Main concept is correct, but the answer introd..."
2,What is forward kinematics?,07_velocity_kinematics.pdf,2,0.3385,Forward kinematics is a function that defines ...,Yes,Yes,Yes,Correct definition of forward kinematics based...
3,What is a rotation matrix?,02_rigid_motion_1.pdf,25,0.3306,A rotation matrix is a mathematical representa...,Yes,Yes,Yes,Correct explanation of a rotation matrix based...
4,How are rotations represented in three dimensi...,02_rigid_motion_1.pdf,11,0.2265,Rotations in three dimensions are represented ...,Yes,Yes,Yes,Correctly identifies the rotation matrix as th...
5,What is velocity kinematics?,07_velocity_kinematics.pdf,2,0.3775,Velocity kinematics refers to the relationship...,Yes,Yes,Yes,Correctly explains that velocity relationships...
6,What is the relationship between joint velocit...,07_velocity_kinematics.pdf,2,0.3785,The relationship between joint velocities and ...,Yes,Yes,Yes,Correctly relates joint velocities and robot m...
7,What are the different types of mobile robot l...,01_intro_to_robotics.pdf,5,0.3302,"According to the retrieved context, there are ...",Yes,Yes,Yes,Correctly lists the locomotion methods explici...
8,Why is the Jacobian important in robot motion?,07_velocity_kinematics.pdf,2,0.2749,The Jacobian is important in robot motion beca...,Yes,No,Yes,Relevant context was retrieved and explicitly ...
9,What problems can occur near a robot singularity?,09_singularities.pdf,3,0.3440,A robot singularity can occur when the workspa...,Yes,Yes,Yes,Correctly identifies the inverse-kinematics pr...


In [93]:
supported_df = evaluation_df[
    evaluation_df["retrieval_relevant"] != "N/A"
]

unsupported_df = evaluation_df[
    evaluation_df["retrieval_relevant"] == "N/A"
]

retrieval_success = (
    supported_df["retrieval_relevant"].eq("Yes").mean() * 100
)

supported_answer_accuracy = (
    supported_df["answer_correct"].eq("Yes").mean() * 100
)

overall_answer_accuracy = (
    evaluation_df["answer_correct"].eq("Yes").mean() * 100
)

unsupported_refusal_rate = (
    unsupported_df["answer_correct"].eq("Yes").mean() * 100
)

fully_grounded_rate = (
    evaluation_df["grounded"].eq("Yes").mean() * 100
)

print(f"Supported retrieval success: {retrieval_success:.1f}%")
print(f"Supported answer accuracy: {supported_answer_accuracy:.1f}%")
print(f"Overall answer accuracy: {overall_answer_accuracy:.1f}%")
print(f"Unsupported-question refusal rate: {unsupported_refusal_rate:.1f}%")
print(f"Fully grounded answers: {fully_grounded_rate:.1f}%")

Supported retrieval success: 100.0%
Supported answer accuracy: 90.0%
Overall answer accuracy: 91.7%
Unsupported-question refusal rate: 100.0%
Fully grounded answers: 91.7%


### Evaluation Results

RoboRAG was evaluated using **12 questions**, including 10 questions supported by the robotics knowledge base and 2 deliberately unsupported questions.

The evaluation showed:

- **Supported-question retrieval success: 100%**
- **Supported-question answer accuracy: 90%**
- **Overall answer accuracy: 91.7%**
- **Unsupported-question refusal rate: 100%**
- **Fully grounded answer rate: 91.7%**

The semantic retrieval component performed strongly across all supported robotics questions. Relevant lecture material was retrieved for every in-domain question.

The two deliberately unsupported questions produced substantially larger retrieval distances than the robotics questions and were correctly rejected by the generation model rather than answered using pretrained external knowledge.

Most generated answers were also directly supported by the retrieved lecture context.

### Failure Analysis and Mitigation

The evaluation revealed that the strongest component of RoboRAG was the semantic retrieval stage. Relevant context was retrieved successfully for all supported robotics questions.

The main observed limitation occurred during answer generation rather than retrieval.

For the question **"Why is the Jacobian important in robot motion?"**, the retriever returned highly relevant context that explicitly stated that the Jacobian is one of the most important quantities in the analysis and control of robot motion. However, the lightweight `llama3.2:latest` model incorrectly returned the fallback response indicating that the information could not be found.

This represents a **generation failure rather than a retrieval failure**. The relevant evidence was available, but the language model did not use it correctly.

A second minor issue was observed in the singularity answer, where the model introduced a small amount of explanatory terminology that was not explicitly present in the retrieved passage. Although the core answer remained correct, this shows that even a grounded prompt cannot completely eliminate unsupported wording.

Mitigation steps included:

- reducing retrieval from Top-4 to **Top-2** to provide the lightweight model with a smaller and more focused context,
- separating system instructions from the user/context message when calling Ollama,
- setting the generation temperature to **0**,
- explicitly instructing the model not to use pretrained knowledge,
- instructing the model to stay close to the terminology contained in the retrieved context,
- and requiring a fixed fallback response when sufficient evidence is unavailable.

Future improvements could include using a stronger local language model, applying a retrieval-distance threshold for out-of-domain questions, adding reranking, and performing more extensive prompt optimization.

## 2.7 Export and Persistence Verification

The final offline artifacts were exported so that the application backend can load the precomputed knowledge base directly without repeating document extraction, chunking, or embedding generation.

The exported artifacts include:

- the persistent ChromaDB vector store,
- the `roborag_chunks` collection,
- all 422 indexed chunks,
- source and page metadata,
- and the RAG configuration file.

The following verification reloads these artifacts from disk to confirm that they are ready for use by the FastAPI backend.

In [94]:
import json
import chromadb

VERIFY_VECTOR_STORE = PROJECT_ROOT / "backend" / "data" / "vector_store"
VERIFY_CONFIG_PATH = PROJECT_ROOT / "backend" / "data" / "rag_config.json"

print("Vector store exists:", VERIFY_VECTOR_STORE.exists())
print("Config exists:", VERIFY_CONFIG_PATH.exists())

Vector store exists: True
Config exists: True


In [95]:
with open(VERIFY_CONFIG_PATH, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

saved_config

{'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'collection_name': 'roborag_chunks',
 'chunk_size': 800,
 'chunk_overlap': 150,
 'embedding_dimension': 384,
 'number_of_documents': 11,
 'number_of_chunks': 422,
 'top_k': 4,
 'ollama_model': 'llama3.2:latest',
 'max_retrieval_distance': 0.65}

In [96]:
verification_client = chromadb.PersistentClient(
    path=str(VERIFY_VECTOR_STORE)
)

verification_collection = verification_client.get_collection(
    name=saved_config["collection_name"]
)

print("Collection:", verification_collection.name)
print("Stored records:", verification_collection.count())

Collection: roborag_chunks
Stored records: 422


In [97]:
verification_sample = verification_collection.peek(limit=1)

print("Sample document:")
print(verification_sample["documents"][0])

print("\nSample metadata:")
print(verification_sample["metadatas"][0])

Sample document:
CSE 432 Robotics
Course Introduction
Ahmed Asker, Ph.D.
Associate Professor in Mechatronic and Robotics
Ahmed.asker@ejust.edu.eg
Egypt-Japan University of Science and Technology
1

Sample metadata:
{'source': '01_intro_to_robotics.pdf', 'chunk_index': 0, 'page': 1}


### Export Verification Results

The persisted RoboRAG knowledge base was successfully reloaded from disk.

Verification confirmed that:

- the ChromaDB vector-store directory exists,
- the RAG configuration file exists,
- the `roborag_chunks` collection can be loaded without rebuilding the documents,
- all **422 vector records** remain available,
- and the original source and page metadata are preserved.

This confirms that the offline indexing pipeline does not need to run during normal application requests.

The FastAPI backend can therefore load the existing vector store during application startup and use it directly for online retrieval.

## 2.6 Final End-to-End Evaluation

The final RoboRAG application is evaluated through the deployed
FastAPI backend so that the results represent the same pipeline used
by the Streamlit frontend.

The evaluation includes 10 robotics questions and 2 unsupported
questions.

In [98]:
import httpx
import pandas as pd

BACKEND_URL = "http://127.0.0.1:8010"

final_questions = [
    "What is the Jacobian in robotics?",
    "What is a robot singularity?",
    "What is forward kinematics?",
    "What is a rotation matrix?",
    "How are rotations represented in three dimensions?",
    "What is velocity kinematics?",
    "What is the relationship between joint velocities and robot motion?",
    "What are the different types of mobile robot locomotion?",
    "Why is the Jacobian important in robot motion?",
    "What problems can occur near a robot singularity?",
    "What was Microsoft's revenue in 2025?",
    "What is the capital city of Australia?",
]

final_results = []

for i, question in enumerate(final_questions, start=1):

    print(f"Running {i}/{len(final_questions)}: {question}")

    response = httpx.post(
        f"{BACKEND_URL}/query",
        json={"question": question},
        timeout=120.0,
    )

    response.raise_for_status()

    data = response.json()

    sources = data.get("sources", [])

    if sources:
        top_source = sources[0]["source"]
        top_page = sources[0]["page"]
        top_distance = sources[0].get("distance")
    else:
        top_source = None
        top_page = None
        top_distance = None

    final_results.append(
        {
            "question": question,
            "answer": data["answer"],
            "top_source": top_source,
            "top_page": top_page,
            "top_distance": top_distance,
            "number_of_sources": len(sources),
        }
    )

print("\nFinal evaluation complete.")

Running 1/12: What is the Jacobian in robotics?
Running 2/12: What is a robot singularity?
Running 3/12: What is forward kinematics?
Running 4/12: What is a rotation matrix?
Running 5/12: How are rotations represented in three dimensions?
Running 6/12: What is velocity kinematics?
Running 7/12: What is the relationship between joint velocities and robot motion?
Running 8/12: What are the different types of mobile robot locomotion?
Running 9/12: Why is the Jacobian important in robot motion?
Running 10/12: What problems can occur near a robot singularity?
Running 11/12: What was Microsoft's revenue in 2025?
Running 12/12: What is the capital city of Australia?

Final evaluation complete.


In [99]:
final_evaluation_df = pd.DataFrame(final_results)

final_evaluation_df

,question,answer,top_source,top_page,top_distance,number_of_sources
0,What is the Jacobian in robotics?,The Jacobian in robotics is a matrix that repr...,07_velocity_kinematics.pdf,2,0.3195,4
1,What is a robot singularity?,A robot singularity refers to a workspace poin...,09_singularities.pdf,3,0.4223,4
2,What is forward kinematics?,Forward kinematics is a function that defines ...,07_velocity_kinematics.pdf,2,0.3385,4
3,What is a rotation matrix?,A rotation matrix is a mathematical representa...,02_rigid_motion_1.pdf,25,0.3306,4
4,How are rotations represented in three dimensi...,Rotations in three dimensions are represented ...,02_rigid_motion_1.pdf,11,0.2265,4
5,What is velocity kinematics?,Velocity kinematics refers to the relationship...,07_velocity_kinematics.pdf,2,0.3775,4
6,What is the relationship between joint velocit...,The relationship between joint velocities and ...,07_velocity_kinematics.pdf,2,0.3785,4
7,What are the different types of mobile robot l...,"According to the retrieved context, there are ...",01_intro_to_robotics.pdf,5,0.3302,4
8,Why is the Jacobian important in robot motion?,The Jacobian is important in robot motion beca...,07_velocity_kinematics.pdf,2,0.2749,4
9,What problems can occur near a robot singularity?,A robot singularity can occur when the workspa...,09_singularities.pdf,3,0.3440,4


In [100]:
for i, row in final_evaluation_df.iterrows():

    print("=" * 100)
    print(f"QUESTION {i + 1}")
    print(row["question"])

    print("\nANSWER:")
    print(row["answer"])

    print("\nTOP SOURCE:")
    print(row["top_source"])

    print("\nPAGE:")
    print(row["top_page"])

    print("\nDISTANCE:")
    print(row["top_distance"])

    print()

QUESTION 1
What is the Jacobian in robotics?

ANSWER:
The Jacobian in robotics is a matrix that represents the vector version of the ordinary derivative of a scalar function.

The Jacobian is a crucial quantity in the analysis and control of robot motion, as it determines the velocity relationships between the space of Cartesian positions and orientations and the space of joint positions. This matrix is derived from the forward kinematic equations, which define a function between the space of Cartesian positions and orientations and the space of joint positions.

TOP SOURCE:
07_velocity_kinematics.pdf

PAGE:
2

DISTANCE:
0.3195

QUESTION 2
What is a robot singularity?

ANSWER:
A robot singularity refers to a workspace point that becomes unreachable under small variations in link parameters, such as length or offsets.

In the context of robotics, a singularity can occur due to the limitations of the robot's kinematic structure, making it difficult or impossible to achieve a specific pos

In [101]:
final_evaluation_df["retrieval_relevant"] = ""
final_evaluation_df["answer_correct"] = ""
final_evaluation_df["grounded"] = ""
final_evaluation_df["notes"] = ""

final_evaluation_df

,question,answer,top_source,top_page,top_distance,number_of_sources,retrieval_relevant,answer_correct,grounded,notes
0,What is the Jacobian in robotics?,The Jacobian in robotics is a matrix that repr...,07_velocity_kinematics.pdf,2,0.3195,4,,,,
1,What is a robot singularity?,A robot singularity refers to a workspace poin...,09_singularities.pdf,3,0.4223,4,,,,
2,What is forward kinematics?,Forward kinematics is a function that defines ...,07_velocity_kinematics.pdf,2,0.3385,4,,,,
3,What is a rotation matrix?,A rotation matrix is a mathematical representa...,02_rigid_motion_1.pdf,25,0.3306,4,,,,
4,How are rotations represented in three dimensi...,Rotations in three dimensions are represented ...,02_rigid_motion_1.pdf,11,0.2265,4,,,,
5,What is velocity kinematics?,Velocity kinematics refers to the relationship...,07_velocity_kinematics.pdf,2,0.3775,4,,,,
6,What is the relationship between joint velocit...,The relationship between joint velocities and ...,07_velocity_kinematics.pdf,2,0.3785,4,,,,
7,What are the different types of mobile robot l...,"According to the retrieved context, there are ...",01_intro_to_robotics.pdf,5,0.3302,4,,,,
8,Why is the Jacobian important in robot motion?,The Jacobian is important in robot motion beca...,07_velocity_kinematics.pdf,2,0.2749,4,,,,
9,What problems can occur near a robot singularity?,A robot singularity can occur when the workspa...,09_singularities.pdf,3,0.3440,4,,,,


In [102]:
manual_evaluation = [
    # Q0 - Jacobian
    (
        "Yes",
        "Yes",
        "Yes",
        "Relevant Jacobian context was retrieved and the answer correctly explains its definition and role in robot motion."
    ),

    # Q1 - Singularity
    (
        "Yes",
        "Yes",
        "Yes",
        "Relevant singularity lecture content was retrieved and the answer correctly describes singularity behavior."
    ),

    # Q2 - Forward kinematics
    (
        "Yes",
        "Yes",
        "Mostly",
        "The core forward-kinematics definition is correct and retrieved successfully, but the generated explanation may include some additional pretrained terminology."
    ),

    # Q3 - Rotation matrix
    (
        "Yes",
        "Yes",
        "Yes",
        "Relevant rigid-motion material was retrieved and the answer correctly explains the rotation matrix."
    ),

    # Q4 - 3D rotations
    (
        "Yes",
        "Yes",
        "Yes",
        "The retrieved lecture directly discusses three-dimensional rotation representation and the answer is consistent with it."
    ),

    # Q5 - Velocity kinematics
    (
        "Yes",
        "Yes",
        "Yes",
        "Relevant velocity-kinematics content was retrieved and the answer correctly describes the relationship through the Jacobian."
    ),

    # Q6 - Joint velocities and robot motion
    (
        "Yes",
        "Yes",
        "Yes",
        "The retrieved context correctly relates joint velocities to robot motion through the Jacobian."
    ),

    # Q7 - Mobile locomotion
    (
        "Yes",
        "Yes",
        "Yes",
        "Relevant robotics/mobile-robot material was retrieved and the answer correctly identifies the locomotion methods."
    ),

    # Q8 - Jacobian importance
    (
        "Yes",
        "Yes",
        "Yes",
        "The retrieved velocity-kinematics lecture explicitly states the importance of the Jacobian in analysis and control of robot motion."
    ),

    # Q9 - Problems near singularity
    (
        "Yes",
        "Yes",
        "Yes",
        "Relevant singularity material was retrieved and the answer correctly describes inverse-kinematics problems near singularities."
    ),

    # Q10 - Microsoft revenue
    (
        "N/A",
        "Yes",
        "Yes",
        "The question is outside the robotics knowledge base and RoboRAG correctly refused to answer."
    ),

    # Q11 - Australia capital
    (
        "N/A",
        "Yes",
        "Yes",
        "The question is outside the robotics knowledge base and RoboRAG correctly refused to answer."
    ),
]


for i, values in enumerate(manual_evaluation):

    final_evaluation_df.loc[
        i,
        [
            "retrieval_relevant",
            "answer_correct",
            "grounded",
            "notes",
        ],
    ] = values


final_evaluation_df

,question,answer,top_source,top_page,top_distance,number_of_sources,retrieval_relevant,answer_correct,grounded,notes
0,What is the Jacobian in robotics?,The Jacobian in robotics is a matrix that repr...,07_velocity_kinematics.pdf,2,0.3195,4,Yes,Yes,Yes,Relevant Jacobian context was retrieved and th...
1,What is a robot singularity?,A robot singularity refers to a workspace poin...,09_singularities.pdf,3,0.4223,4,Yes,Yes,Yes,Relevant singularity lecture content was retri...
2,What is forward kinematics?,Forward kinematics is a function that defines ...,07_velocity_kinematics.pdf,2,0.3385,4,Yes,Yes,Mostly,The core forward-kinematics definition is corr...
3,What is a rotation matrix?,A rotation matrix is a mathematical representa...,02_rigid_motion_1.pdf,25,0.3306,4,Yes,Yes,Yes,Relevant rigid-motion material was retrieved a...
4,How are rotations represented in three dimensi...,Rotations in three dimensions are represented ...,02_rigid_motion_1.pdf,11,0.2265,4,Yes,Yes,Yes,The retrieved lecture directly discusses three...
5,What is velocity kinematics?,Velocity kinematics refers to the relationship...,07_velocity_kinematics.pdf,2,0.3775,4,Yes,Yes,Yes,Relevant velocity-kinematics content was retri...
6,What is the relationship between joint velocit...,The relationship between joint velocities and ...,07_velocity_kinematics.pdf,2,0.3785,4,Yes,Yes,Yes,The retrieved context correctly relates joint ...
7,What are the different types of mobile robot l...,"According to the retrieved context, there are ...",01_intro_to_robotics.pdf,5,0.3302,4,Yes,Yes,Yes,Relevant robotics/mobile-robot material was re...
8,Why is the Jacobian important in robot motion?,The Jacobian is important in robot motion beca...,07_velocity_kinematics.pdf,2,0.2749,4,Yes,Yes,Yes,The retrieved velocity-kinematics lecture expl...
9,What problems can occur near a robot singularity?,A robot singularity can occur when the workspa...,09_singularities.pdf,3,0.3440,4,Yes,Yes,Yes,Relevant singularity material was retrieved an...


In [103]:
pd.set_option(
    "display.max_colwidth",
    120
)

display(
    final_evaluation_df[
        [
            "question",
            "top_source",
            "top_page",
            "top_distance",
            "retrieval_relevant",
            "answer_correct",
            "grounded",
            "notes",
        ]
    ]
)

,question,top_source,top_page,top_distance,retrieval_relevant,answer_correct,grounded,notes
0,What is the Jacobian in robotics?,07_velocity_kinematics.pdf,2,0.3195,Yes,Yes,Yes,Relevant Jacobian context was retrieved and the answer correctly explains its definition and role in robot motion.
1,What is a robot singularity?,09_singularities.pdf,3,0.4223,Yes,Yes,Yes,Relevant singularity lecture content was retrieved and the answer correctly describes singularity behavior.
2,What is forward kinematics?,07_velocity_kinematics.pdf,2,0.3385,Yes,Yes,Mostly,"The core forward-kinematics definition is correct and retrieved successfully, but the generated explanation may incl..."
3,What is a rotation matrix?,02_rigid_motion_1.pdf,25,0.3306,Yes,Yes,Yes,Relevant rigid-motion material was retrieved and the answer correctly explains the rotation matrix.
4,How are rotations represented in three dimensions?,02_rigid_motion_1.pdf,11,0.2265,Yes,Yes,Yes,The retrieved lecture directly discusses three-dimensional rotation representation and the answer is consistent with...
5,What is velocity kinematics?,07_velocity_kinematics.pdf,2,0.3775,Yes,Yes,Yes,Relevant velocity-kinematics content was retrieved and the answer correctly describes the relationship through the J...
6,What is the relationship between joint velocities and robot motion?,07_velocity_kinematics.pdf,2,0.3785,Yes,Yes,Yes,The retrieved context correctly relates joint velocities to robot motion through the Jacobian.
7,What are the different types of mobile robot locomotion?,01_intro_to_robotics.pdf,5,0.3302,Yes,Yes,Yes,Relevant robotics/mobile-robot material was retrieved and the answer correctly identifies the locomotion methods.
8,Why is the Jacobian important in robot motion?,07_velocity_kinematics.pdf,2,0.2749,Yes,Yes,Yes,The retrieved velocity-kinematics lecture explicitly states the importance of the Jacobian in analysis and control o...
9,What problems can occur near a robot singularity?,09_singularities.pdf,3,0.3440,Yes,Yes,Yes,Relevant singularity material was retrieved and the answer correctly describes inverse-kinematics problems near sing...


In [104]:
supported_df = final_evaluation_df[
    final_evaluation_df["retrieval_relevant"] != "N/A"
]

unsupported_df = final_evaluation_df[
    final_evaluation_df["retrieval_relevant"] == "N/A"
]


retrieval_success = (
    supported_df["retrieval_relevant"]
    .eq("Yes")
    .mean()
    * 100
)


supported_answer_accuracy = (
    supported_df["answer_correct"]
    .eq("Yes")
    .mean()
    * 100
)


overall_answer_accuracy = (
    final_evaluation_df["answer_correct"]
    .eq("Yes")
    .mean()
    * 100
)


fully_grounded_rate = (
    final_evaluation_df["grounded"]
    .eq("Yes")
    .mean()
    * 100
)


unsupported_refusal_rate = (
    unsupported_df["answer_correct"]
    .eq("Yes")
    .mean()
    * 100
)


print(
    f"Supported retrieval success: "
    f"{retrieval_success:.1f}%"
)

print(
    f"Supported answer accuracy: "
    f"{supported_answer_accuracy:.1f}%"
)

print(
    f"Overall answer accuracy: "
    f"{overall_answer_accuracy:.1f}%"
)

print(
    f"Fully grounded answer rate: "
    f"{fully_grounded_rate:.1f}%"
)

print(
    f"Unsupported-question refusal rate: "
    f"{unsupported_refusal_rate:.1f}%"
)

Supported retrieval success: 100.0%
Supported answer accuracy: 100.0%
Overall answer accuracy: 100.0%
Fully grounded answer rate: 91.7%
Unsupported-question refusal rate: 100.0%


In [105]:
evaluation_output_path = (
    PROJECT_ROOT
    / "backend"
    / "data"
    / "final_evaluation.csv"
)

final_evaluation_df.to_csv(
    evaluation_output_path,
    index=False
)

print(
    "Saved final evaluation to:",
    evaluation_output_path
)

Saved final evaluation to: c:\Users\yasser\Projects\rag-study-assistant\backend\data\final_evaluation.csv


### Final Evaluation Summary

The final RoboRAG configuration was evaluated using 12 questions:
10 robotics questions supported by the indexed course material and
2 deliberately unsupported questions.

The final manually reviewed results were:

- **Supported retrieval success:** 100.0%
- **Supported answer accuracy:** 100.0%
- **Overall answer accuracy:** 100.0%
- **Fully grounded answer rate:** 91.7%
- **Unsupported-question refusal rate:** 100.0%

The retrieval pipeline successfully returned relevant course material
for all supported robotics questions.

Both unsupported questions were correctly rejected rather than answered
using unrelated external knowledge.

One answer was classified as mostly grounded rather than fully grounded
because the generated explanation included some additional terminology
beyond the retrieved wording. This demonstrates that the LLM generation
stage can still introduce minor unsupported elaboration even when the
core answer is correct.

The final application uses:

- `sentence-transformers/all-MiniLM-L6-v2`
- ChromaDB with cosine-distance retrieval
- Top-K = 4
- `llama3.2:latest` through Ollama
- Maximum retrieval distance = 0.65
- FastAPI backend
- Streamlit frontend